# Volume 1, Chapter 10: Gaussian and Eisenstein Integers (Python companion)

Exercises first, solutions at the end. Computer-science threads: **exact arithmetic in $\Z[i]$** with integer pairs
(no floating point), **factorisation** and the check of Jacobi's $r_2$ formula, the **Gaussian gcd** as a fast way to
write a large prime as a sum of two squares, and a **graph search** on Gaussian primes (the Gaussian moat problem).

In [ ]:
import math
import random
from collections import deque
import matplotlib.pyplot as plt
from sympy import factorint, isprime, randprime, sqrt_mod, divisors
# Reproducible teaching examples; these seeds are not for cryptographic keys.
import random
from sympy.core.random import seed as seed_sympy
random.seed(10110)
seed_sympy(20110)


## Before the experiment: Arithmetic on graph paper

Guess the Gaussian gcd of two lattice points, remembering it is only defined up to a unit. Why do four rotated answers all count as the same answer? All division here uses integers; floating-point rounding is not invited to this lattice party.

**How to use this notebook.** Try the exercise cells first; they are intentionally blank. Then run the solutions in order in a fresh kernel. Assertions check exact identities or a stated finite range; plots and random trials supply evidence, not proofs.


## Exercises

**P10.1 (★) Exact Gaussian arithmetic.** Represent $a+bi$ as a pair of Python integers. Write `gdivmod(x, y)` (Euclidean division with rounding, Proposition 10.2) and `ggcd(x, y)` without any float, and check that the remainder always has smaller norm. Recompute $\gcd(11+3i,1+8i)$ and $\gcd(1000+3i,972+235i)$.

In [ ]:
# Your code here


**P10.2 (★★) Factorisation and $r_2$.** Write `gfactor(z)` returning the Gaussian prime factorisation of $z$ (factor $N(z)$ in $\Z$, split each $p\equiv1\pmod4$, then find the valuations by trial division). Use it to list all $z$ with $N(z)=n$ and check Theorem 10.5 for $n\le3000$.

In [ ]:
# Your code here


**P10.3 (★) Pictures.** Plot the Gaussian primes with norm $\le 5000$, and the Eisenstein primes $a+b\omega$ with norm $\le 3000$ (at their true position in $\C$). Which symmetries do you see, and why (units, conjugation)?

In [ ]:
# Your code here


**P10.4 (★★) CS application: two squares for a 512-bit prime.** Using Exercise 10.3, write $p=x^2+y^2$ for random 512-bit primes $p\equiv1\pmod4$ with a Gaussian gcd, and compare the running time with Cornacchia's algorithm (Chapter 7's notebook).

*How to think about it:* both are Euclid's algorithm in disguise; Cornacchia runs it on $(p,a)$ in $\Z$ and stops at $\sqrt p$, while the Gaussian version runs it on $(p,a+i)$ in $\Z[i]$ to the end. Expect the same $O(\log p)$ number of steps.

In [ ]:
# Your code here


**P10.5 (★★★) The Gaussian moat.** Can one walk to infinity in the first quadrant, stepping only on Gaussian primes, with steps of length at most $k$? Build the graph of Gaussian primes $a+bi$ with $a,b\ge0$ and $a^2+b^2\le R^2$ ($R=1000$), join primes at distance $\le k$, and find the farthest prime reachable from $1+i$ for $k=\sqrt2,2,\sqrt8,\sqrt{10}$.

*How to think about it:* breadth-first search over a graph whose edges are found by scanning the few lattice points within distance $k$; the question of whether some finite $k$ lets you escape is open (the *Gaussian moat problem*), but for small $k$ moats are found by computation.

In [ ]:
# Your code here


---
## Solutions

### P10.1
Rounding $\frac uN$ to the nearest integer with integers only: `(2*u + N) // (2*N)`.

In [ ]:
def gmul(x, y):
    (a, b), (c, d) = x, y
    return (a * c - b * d, a * d + b * c)

def gnorm(x):
    return x[0] ** 2 + x[1] ** 2

def rdiv(u, n):                     # nearest integer to u/n, n > 0
    return (2 * u + n) // (2 * n)

def gdivmod(x, y):
    n = gnorm(y)
    num = gmul(x, (y[0], -y[1]))    # x * conj(y)
    q = (rdiv(num[0], n), rdiv(num[1], n))
    qy = gmul(q, y)
    return q, (x[0] - qy[0], x[1] - qy[1])

def ggcd(x, y):
    while y != (0, 0):
        _, r = gdivmod(x, y)
        x, y = y, r
    return x

random.seed(0)
for _ in range(10000):
    x = (random.randint(-10**9, 10**9), random.randint(-10**9, 10**9))
    y = (random.randint(-10**6, 10**6), random.randint(-10**6, 10**6))
    if y != (0, 0):
        q, r = gdivmod(x, y)
        assert 2 * gnorm(r) <= gnorm(y)          # in fact N(r) <= N(y)/2
print(ggcd((11, 3), (1, 8)), ggcd((1000, 3), (972, 235)))

### P10.2

In [ ]:
UNITS = [(1, 0), (0, 1), (-1, 0), (0, -1)]

def split_prime(p):
    if p == 2:
        return (1, 1)
    a = sqrt_mod(-1, p)
    return ggcd((p, 0), (a, 1))

def gdivides(d, z):
    q, r = gdivmod(z, d)
    return r == (0, 0), q

def gfactor(z):
    if z == (0, 0):
        raise ValueError("zero has no finite prime factorisation")
    out = {}
    for p, e in factorint(gnorm(z)).items():
        cands = [(p, 0)] if p % 4 == 3 else ([split_prime(p)] if p == 2 else
                 [split_prime(p), (split_prime(p)[0], -split_prime(p)[1])])
        for pi in cands:
            while True:
                ok, q = gdivides(pi, z)
                if not ok:
                    break
                out[pi] = out.get(pi, 0) + 1
                z = q
    return z, out                     # z is now a unit

print(gfactor((2026, 0)), gfactor((65, 0)))

def r2_brute(n):
    m = math.isqrt(n)
    return sum(1 for x in range(-m, m + 1) for y in range(-m, m + 1) if x * x + y * y == n)

chi4 = lambda d: 0 if d % 2 == 0 else (1 if d % 4 == 1 else -1)
assert all(r2_brute(n) == 4 * sum(chi4(d) for d in divisors(n)) for n in range(1, 3001))
print("Theorem 10.5 checked for n <= 3000")
for test_z in [(1, 0), (2, 1), (6, 7), (-11, 5), (65, 0), (0, -12)]:
    unit, factors = gfactor(test_z)
    rebuilt = unit
    for prime, exponent in factors.items():
        for _ in range(exponent):
            rebuilt = gmul(rebuilt, prime)
    assert rebuilt == test_z and gnorm(unit) == 1


### P10.3
Both pictures are invariant under multiplication by units (rotations by $90^\circ$, resp. $60^\circ$) and under conjugation (reflection in the real axis), because these maps send irreducibles to irreducibles.

In [ ]:
def is_gauss_prime(a, b):
    if a == 0 or b == 0:
        c = abs(a + b)
        return isprime(c) and c % 4 == 3
    return isprime(a * a + b * b)

R = int(math.isqrt(5000))
pts = [(a, b) for a in range(-R, R + 1) for b in range(-R, R + 1) if a * a + b * b <= 5000 and is_gauss_prime(a, b)]
fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
ax[0].scatter(*zip(*pts), s=2); ax[0].set_aspect("equal"); ax[0].set_title("Gaussian primes")

def is_eis_prime(a, b):
    n = a * a - a * b + b * b
    if isprime(n):
        return True
    # rational primes q = 2 mod 3 up to associates: norm q^2
    q = math.isqrt(n)
    return q * q == n and isprime(q) and q % 3 == 2

E = [(a, b) for a in range(-70, 71) for b in range(-70, 71) if 0 < a * a - a * b + b * b <= 3000 and is_eis_prime(a, b)]
ax[1].scatter([a - b / 2 for a, b in E], [b * math.sqrt(3) / 2 for a, b in E], s=2, c="tab:red")
ax[1].set_aspect("equal"); ax[1].set_title("Eisenstein primes"); plt.show()

For an element of norm $q^2$ with $q\equiv2\pmod3$ prime, unique factorisation forces it to be an associate of $q$ (its only irreducible factors divide $q$, which is irreducible), so the test above is exact.

### P10.4

In [ ]:
import time

def cornacchia(p):
    a = sqrt_mod(-1, p)
    if a < p // 2:
        a = p - a
    r0, r1, s = p, a, math.isqrt(p)
    while r1 > s:
        r0, r1 = r1, r0 % r1
    return r1, math.isqrt(p - r1 * r1)

def two_squares_gauss(p):
    a = sqrt_mod(-1, p)
    x, y = ggcd((p, 0), (a, 1))
    return abs(x), abs(y)

random.seed(1)
ps = []
while len(ps) < 200:
    p = randprime(2**511, 2**512)
    if p % 4 == 1:
        ps.append(p)
roots = {p: sqrt_mod(-1, p) for p in ps}       # shared cost, excluded below
t0 = time.time()
for p in ps:
    x, y = ggcd((p, 0), (roots[p], 1)); assert x * x + y * y == p
t1 = time.time()
for p in ps:
    a = roots[p] if roots[p] > p // 2 else p - roots[p]
    r0, r1, s = p, a, math.isqrt(p)
    while r1 > s:
        r0, r1 = r1, r0 % r1
    assert r1 * r1 + math.isqrt(p - r1 * r1) ** 2 == p
t2 = time.time()
print(f"Gaussian gcd: {1000 * (t1 - t0) / len(ps):.3f} ms per prime,  Cornacchia: {1000 * (t2 - t1) / len(ps):.3f} ms")

Both are fast; Cornacchia is a few times quicker because it works with single integers and stops halfway. The expensive part of both is finding $\sqrt{-1}\bmod p$ (Tonelli–Shanks, Chapter 6).

### P10.5

In [ ]:
R = 1000
primes = {(a, b) for a in range(R + 1) for b in range(R + 1)
          if a * a + b * b <= R * R and is_gauss_prime(a, b)}

def farthest(k2):
    k = math.isqrt(k2)
    offs = [(dx, dy) for dx in range(-k, k + 1) for dy in range(-k, k + 1)
            if 0 < dx * dx + dy * dy <= k2]
    start = (1, 1)
    seen, q = {start}, deque([start])
    while q:
        a, b = q.popleft()
        for dx, dy in offs:
            w = (a + dx, b + dy)
            if w in primes and w not in seen:
                seen.add(w); q.append(w)
    return max(seen, key=gnorm), len(seen)

for k2 in [2, 4, 8, 10]:
    far, size = farthest(k2)
    print(f"step <= sqrt({k2}): {size} primes reached, farthest {far}, |z| = {math.sqrt(gnorm(far)):.1f}"
          + ("  (hits the boundary R)" if gnorm(far) > (R - 3) ** 2 else ""))

For small step sizes the walk is trapped by a moat close to the origin. Gethner, Wagon and Wick (1998) showed by computation that steps of length $\sqrt{26}$ are not enough either; whether some finite step size escapes to infinity is unknown.

A path that hits the boundary of our search disk may continue outside it. Conversely, an apparently closed component is certified globally only if every permitted neighboring lattice point has been checked, including neighbors just outside the disk. The output describes this finite induced graph; it is not a proof about an infinite walk.


## One more question before closing

Factor the same Gaussian integer after multiplying it by i. Which factors change, and which information is invariant?

Write one sentence identifying what was proved in the chapter, what was checked exactly by the code, and what remains a conjecture or a heuristic.
